# Finding most recent sequences from full dataset

* California Cow (B3.13) most recent sequence
* Idaho Cow (B3.13) most recent sequence
* Nevada Cow (D1.1) most recent sequence

In [5]:
import os
import pandas as pd

home = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/"
dataset_new_d1_1 = home + "Combinations/GISAID_Andersen_NCBI_Virus/04-14-2025--05-14-2025_D1_1/"
dataset_new_b3_13 = home + "Combinations/GISAID_Andersen_NCBI_Virus/04-14-2025--05-14-2025_B3_13/"
dataset_old = home + "Combinations/GISAID_Andersen_NCBI_Virus/all_genotypes_11-01-2021--04-14-2025/"
states = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu/references/"

os.chdir(states)
states_ref = pd.read_csv("states_ref.csv")

In [ ]:
# Function to prepare dataframes
def fasta_df(file_name, state_ref):

    fasta = pd.DataFrame()
    headers = []
    isolate_ids = []
    isolate_names = []
    subtypes = []
    segments = []
    collection_dates = []
    sequences = []
    host_types = []
    species = []
    genotypes = []
    with open(file_name) as f:
        lines = f.readlines()
        for num, line in enumerate(lines):
            # print(line)
            if line[0] == ">": # If it's a header
                if line[1:].strip() not in headers: # And the previous line is not a header we've seen before
                    header = line[1:].strip() # Remove the ">"
                    # print(header)
                    split_header = header.split("|")
                    split_first_header = split_header[0].split("/")
                    # print(split_header)
                    headers.append(header) 
                    isolate_ids.append(split_first_header[3])
                    isolate_names.append(split_header[0]) # We'll need to extract data from this too
                    # print(split_header[2].split("_")[-1])
                    subtypes.append(split_header[1])  # Get only H5N1
                    genotypes.append(split_header[-1])
                    segments.append(file_name.split("_")[-3])
                    host_types.append(split_header[-2])
                    species.append(split_first_header[1])
                    # if split_header[4] == "2024-01-01":
                    #     collection_dates.append("2024") # No samples were collected 1/1/2024, these are all unknown 
                    # elif split_header[4] == "2025-01-01":
                    #     collection_dates.append("2025")
                    # else: 
                    collection_dates.append(split_header[3].split("_")[-1])
                    if num < len(lines): # If we're not at the last line
                        # for i, l in enumerate(lines[num + 1:]):
                        i = num
                        sequence = ""
                        # print(lines[i])
                        # print(lines[i + 1])
                        while i < len(lines) - 1 and lines[i + 1][0] != ">": # While the next line is part of a sequence
                            sequence = sequence + lines[i + 1].strip()
                            i += 1
                        sequences.append(sequence) # Add next line to sequences
        f.close()

    # Create columns for data frame 
    fasta["Header"] = headers
    fasta["Isolate_Id"] = isolate_ids
    fasta["Isolate_Name"] = isolate_names
    fasta["Subtype"] = subtypes
    fasta["Segment"] = segments
    # Geo_Location is more complicated
    fasta["Geo_Location"] = fasta["Header"].apply(lambda x: state_ref.loc[state_ref["Abbreviation"] == x.split("/")[2], 'Country'].iloc[0] + "-" + x.split("/")[2] if x.split("/")[2] in state_ref["Abbreviation"].values else state_ref.loc[state_ref["State"] == x.split("/")[2].replace("_", " "), 'Country'].iloc[0] + "-" + state_ref.loc[state_ref["State"] == x.split("/")[2].replace("_", " "), 'Abbreviation'].iloc[0] if x.split("/")[2].replace("_", " ") in state_ref["State"].values else x.split("/")[2].replace(": ", "-"))
    fasta["Date Collected"] = collection_dates
    fasta["Species"] = species
    fasta["Host_Type"] = host_types
    fasta["Genotype"] = genotypes
    fasta["Sequence"] = sequences
    
    return fasta



                                                Header  \
0    A/CATTLE/USA/25-006243-002/2025|H5N1|USA|2025|...   
1    A/CATTLE/USA/25-006240-005/2025|H5N1|USA|2025|...   
2    A/CATTLE/USA/25-006031-002/2025|H5N1|USA|2025|...   
3    A/CATTLE/USA/25-006954-002/2025|H5N1|USA|2025|...   
4    A/CATTLE/USA/25-006276-004/2025|H5N1|USA|2025|...   
..                                                 ...   
488  A/bos taurus/UT/24-032712-001-original/2024|H5...   
489  A/bos taurus/UT/24-034800-001-original/2024|H5...   
490  A/bos taurus/UT/24-035672-001-original/2024|H5...   
491  A/bos taurus/UT/24-035680-001-original/2024|H5...   
492  A/bos taurus/UT/25-000076-001-original/2024|H5...   

                 Isolate_Id                                 Isolate_Name  \
0             25-006243-002              A/CATTLE/USA/25-006243-002/2025   
1             25-006240-005              A/CATTLE/USA/25-006240-005/2025   
2             25-006031-002              A/CATTLE/USA/25-006031-002/2025   

In [29]:
os.chdir(dataset_new_b3_13)

new_b3_13_ha = fasta_df("B3.13_HA_combined_04-14-2025--05-14-2025.fasta", states_ref)
new_b3_13_ha_cattle_ca = new_b3_13_ha[(new_b3_13_ha["Host_Type"] == "cattle") & (new_b3_13_ha["Geo_Location"] == "USA-CA")]
# new_b3_13_ha_cattle_id = new_b3_13_ha[(new_b3_13_ha["Host_Type"] == "cattle") & (new_b3_13_ha["Geo_Location"] == "USA-ID")]

os.chdir(dataset_old) 
print(os.listdir())

old_b3_13_ha = fasta_df("all_genotypes_11-01-2021--04-14-2025/B3.13_HA_combined_04-14-2025.fasta", states_ref)
old_b3_13_ha_cattle_id = old_b3_13_ha[(old_b3_13_ha["Host_Type"] == "cattle") & (old_b3_13_ha["Geo_Location"] == "USA-ID")]
old_b3_13_ha_cattle_id["True_Date"] = old_b3_13_ha_cattle_id["Header"].apply(lambda x: x.split("|")[2])

print(new_b3_13_ha_cattle_ca.sort_values(by="Date Collected"))
print(old_b3_13_ha_cattle_id.sort_values(by="True_Date"))

old_d1_1_ha = fasta_df("all_genotypes_11-01-2021--04-14-2025/D1.1_HA_combined_04-14-2025.fasta", states_ref)
old_d1_1_ha_cattle_nv = old_d1_1_ha[(old_d1_1_ha["Host_Type"] == "cattle") & (old_d1_1_ha["Geo_Location"] == "USA-NV")]
old_d1_1_ha_cattle_nv["True_Date"] = old_d1_1_ha_cattle_nv["Header"].apply(lambda x: x.split("|")[2])

print(old_d1_1_ha_cattle_nv.sort_values(by="True_Date"))

os.chdir(dataset_new_d1_1)
new_d1_1_ha = fasta_df("D1.1_HA_combined_04-14-2025--05-14-2025.fasta", states_ref)
new_d1_1_ha_cattle_nv = new_d1_1_ha[(new_d1_1_ha["Host_Type"] == "cattle") & (new_d1_1_ha["Geo_Location"] == "USA-NV")]
new_d1_1_ha_cattle_nv["True_Date"] = new_d1_1_ha_cattle_nv["Header"].apply(lambda x: x.split("|")[2])


print(new_d1_1_ha_cattle_nv.sort_values(by="True_Date"))

# new_b3_13_ha["Geo_Location"] = new_b3_13_ha["full_header"].apply(lambda x: x.split("|")[2])
# new_b3_13_ha["Date"] = new_b3_13_ha["full_header"].apply(lambda x: x.split("|")[3])
# new_b3_13_ha["Host"] = new_b3_13_ha["full_header"].apply(lambda x: x.split("|")[-2])

# print(new_b3_13_ha)

['all_genotypes_11-01-2021--04-14-2025']


C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\3\ipykernel_20200\3994421946.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  old_b3_13_ha_cattle_id["True_Date"] = old_b3_13_ha_cattle_id["Header"].apply(lambda x: x.split("|")[2])


                                                Header  \
206  A/dairy_cow/California/24_033172-001/2022|H5N1...   
207  A/dairy_cow/California/24_033172-002/2022|H5N1...   
208  A/dairy_cow/California/24_033172-003/2022|H5N1...   
321  A/bos taurus/CA/24-035207-002-original/2024|H5...   
320  A/bos taurus/CA/24-035207-001-original/2024|H5...   
..                                                 ...   
483  A/bos taurus/CA/25-010012-003-original/2025|H5...   
480  A/bos taurus/CA/25-009855-001-original/2025|H5...   
481  A/bos taurus/CA/25-009855-002-original/2025|H5...   
482  A/bos taurus/CA/25-009855-003-original/2025|H5...   
484  A/bos taurus/CA/25-010022-002-original/2025|H5...   

                 Isolate_Id                                 Isolate_Name  \
206           24_033172-001    A/dairy_cow/California/24_033172-001/2022   
207           24_033172-002    A/dairy_cow/California/24_033172-002/2022   
208           24_033172-003    A/dairy_cow/California/24_033172-003/2022   

C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\3\ipykernel_20200\3994421946.py:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_d1_1_ha_cattle_nv["True_Date"] = new_d1_1_ha_cattle_nv["Header"].apply(lambda x: x.split("|")[2])
